# Lab 4.3 · Gram–Schmidt and Reduced QR

Explain each entry of Q and R, check orthonormality separately from reconstruction, and connect the QR projection to least squares.

Wooseok Ha, MAS110 Lecture 4 (Fall 2026), lec04.pdf: printed slides 34–37, 47, 49–51 (PDF pages 39–42, 55, 57–59). This is the same A used for projection and least squares in 4.4.

Maintained in [MAS110 Practice Sessions](https://github.com/statchan1106/mas110-practice-sessions). Course companion to [Foundations of LADS](https://github.com/kyunghyuncho/Foundations_of_LADS) by Wanmo Kang and Kyunghyun Cho.

Run all cells in order. NumPy handles the calculations; the final plot uses Matplotlib. Both are available in Colab.

The matrix A has m = 3 rows and n = 2 independent columns. Gram–Schmidt replaces its columns by an orthonormal basis for the same column space; R records the coefficients needed to reconstruct the original columns. The lecture calls the normalized Gram–Schmidt vectors wᵢ before naming the QR columns qᵢ. We consistently use qᵢ for normalized directions and wⱼ for an unnormalized remainder.

## Notation

- **⟨u,v⟩; ‖u‖; u ⊥ v**: An inner product gives one real number, ‖u‖ = √⟨u,u⟩ is its induced norm, and u ⊥ v means ⟨u,v⟩ = 0. The lecture writes |u| for this norm. In Euclidean space the default is uᵀv and the norm is ‖u‖₂; for polynomials it is the stated integral norm.

- **φ; θ; θ̂**: This chapter uses φ for an angle and θ for model coefficients in Lab 4.4. The lecture sometimes uses θ for the angle too; the distinction here prevents mixing an angle with a coefficient vector. A hat marks a minimizing coefficient, not a unit vector.

- **A; G; Q; R; P**: A stores input columns in the Euclidean QR and least-squares labs. G denotes a Gram matrix of inner products, including the polynomial metric (which the lecture locally calls A). Q stores orthonormal columns, R stores their reconstruction coefficients, and P projects onto a subspace. These matrices have different jobs and shapes.

- **A = [a₁ | a₂] ∈ ℝ³ˣ²**: aⱼ here means column j, a vector in ℝ³. In the statistical model of 4.4, the lecture also names a row feature vector aᵢ via row aᵢᵀ. Check the index and shape: columns are feature directions in measurement space; rows collect features for one measurement.

- **wⱼ; qⱼ = wⱼ/‖wⱼ‖₂**: wⱼ is what remains after removing all earlier orthonormal components of aⱼ. qⱼ has unit length; wⱼ generally does not. This normalization needs a nonzero remainder. Independent columns guarantee it in exact arithmetic.

- **rᵢⱼ = qᵢᵀaⱼ (i < j); rⱼⱼ = ‖wⱼ‖₂**: These scalars are the reconstruction coefficients. With the positive-diagonal Gram–Schmidt convention, rⱼⱼ > 0. A library can use different matching signs; that does not change QR or the column space.

- **Q ∈ ℝᵐˣⁿ; R ∈ ℝⁿˣⁿ, m ≥ n**: For full column rank, reduced Q has n orthonormal columns and reduced R is invertible and upper triangular. QᵀQ = Iₙ. For m > n, Q is rectangular and QQᵀ = P, not Iₘ.

- **Complete QR; Q_full ∈ ℝᵐˣᵐ**: Complete Q extends the basis to all of ℝᵐ; R_full has shape m × n. The extra Q columns do not all belong to Col(A). Q_fullQ_fullᵀ = Iₘ, whereas projection onto Col(A) uses only the relevant column-space basis.

- **ε; numerical threshold; matrix rank**: ε is a small parameter in the near-dependent example, not a residual vector. In floating point a tiny remainder can indicate numerical dependence. The demonstration routine compares against a threshold scaled by the matrix norm; it is a teaching rank policy, not an exact symbolic rank test.

- **np.linalg.qr(A, mode="reduced")**: This returns a reduced factorization using LAPACK routines. Rank-deficient matrices still have QR factorizations, but R is singular and extra Q directions need not lie in Col(A). The full-column-rank solve Rθ = Qᵀb then cannot be used as an invertible solve.

## Key ideas

### Gram–Schmidt

Remove each old orthonormal component, then normalize the new information.

wⱼ = aⱼ − Σᵢ<ⱼ(qᵢᵀaⱼ)qᵢ

Keep in mind: A zero remainder cannot be normalized.

### Reduced QR

An orthonormal column basis and the triangular coefficients reconstruct the input.

A = QR; QᵀQ = Iₙ

Keep in mind: Reduced Q is rectangular when m > n.

### Two numerical checks

Measure reconstruction separately from orthonormality.

‖A−QR‖; ‖QᵀQ−I‖

Keep in mind: One small error does not establish the other.

## 1. Name the input columns

Column j is an ambient vector aⱼ, not a row feature vector. Python uses 0-based column indices.

**Predict first:** Why do a1 and a2 each have three entries?

- Line 1, `import numpy as np`: Loads NumPy. Shape: module. Operation: np supplies vector and matrix operations.

- Line 2, `A = np.array([[-1., 1.], [0., 1.], [1., 0.]])`: Stores the lecture matrix. Shape: A: (3, 2). Operation: Its rows are [−1,1], [0,1], [1,0].

- Line 3, `a1, a2 = A[:, 0], A[:, 1]`: Selects its two columns. Shape: two slices → (3,), (3,). Operation: a₁ = (−1,0,1)ᵀ, a₂ = (1,1,0)ᵀ.

In [ ]:
import numpy as np
A = np.array([[-1., 1.], [0., 1.], [1., 0.]])
a1, a2 = A[:, 0], A[:, 1]

In [ ]:
print("A =", A, "a1 =", a1, "a2 =", a2, sep="\n")

**Read the result:** Their dot product is −1, so we must subtract an overlapping component.

a₁ᵀa₂ = −1



## 2. Normalize the first direction

The first column is already nonzero. Its norm becomes the first diagonal entry of R.

**Predict first:** Why must r11 be checked before dividing?

- Line 1, `r11 = np.linalg.norm(a1)`: Measures the first column. Shape: (3,) → scalar. Operation: r₁₁ = √2 > 0.

- Line 2, `q1 = a1 / r11`: Makes the first unit column. Shape: (3,) / scalar → (3,). Operation: q₁ = (−1,0,1)ᵀ/√2.

In [ ]:
r11 = np.linalg.norm(a1)
q1 = a1 / r11

In [ ]:
print("r11 =", r11, "q1 =", q1)

**Read the result:** There is nothing to subtract at the first step. The first input reconstructs as a₁ = r₁₁q₁.

r₁₁ = √2; q₁ᵀq₁ = 1



## 3. Subtract the part of a₂ along q₁

Projection onto the unit direction q₁ uses one dot product. The remainder is perpendicular to q₁.

**Predict first:** What does the negative r12 say about the component’s direction?

- Line 1, `r12 = q1 @ a2`: Finds the old-basis coordinate. Shape: (3,) @ (3,) → scalar. Operation: r₁₂ = −1/√2.

- Line 2, `old_component = r12 * q1`: Turns the scalar coordinate into a vector. Shape: scalar × (3,) → (3,). Operation: The old component is (1/2,0,−1/2)ᵀ.

- Line 3, `w2 = a2 - old_component`: Removes that component. Shape: (3,) − (3,) → (3,). Operation: w₂ = (1/2,1,1/2)ᵀ, perpendicular to q₁.

- Line 4, `r22 = np.linalg.norm(w2)`: Measures the new direction. Shape: (3,) → scalar. Operation: r₂₂ = √6/2 > 0, confirming new information.

In [ ]:
r12 = q1 @ a2
old_component = r12 * q1
w2 = a2 - old_component
r22 = np.linalg.norm(w2)

In [ ]:
print("r12 =", r12, "old component =", old_component, "w2 =", w2, "r22 =", r22, sep="\n")

**Read the result:** The second column splits into an old component plus a nonzero perpendicular remainder.

a₂ = r₁₂q₁ + w₂; q₁ᵀw₂ = 0



## 4. Normalize the remainder and assemble the factors

Normalizing w₂ gives q₂. Storing q₁,q₂ as columns fixes the shape of Q; dot products with A give the entries of R.

**Predict first:** Which entries of R must be zero?

- Line 1, `q2 = w2 / r22`: Makes the second unit column. Shape: (3,) / scalar → (3,). Operation: q₂ = (1,2,1)ᵀ/√6.

- Line 2, `Q = np.column_stack((q1, q2))`: Stores unit directions as columns. Shape: two (3,) arrays → Q: (3, 2). Operation: QᵀQ = I₂.

- Line 3, `R = Q.T @ A`: Reads the original columns’ coordinates. Shape: (2, 3) @ (3, 2) → R: (2, 2). Operation: R = [[√2,−1/√2],[0,√6/2]], up to rounding.

In [ ]:
q2 = w2 / r22
Q = np.column_stack((q1, q2))
R = Q.T @ A

In [ ]:
print("Q =", Q, "R =", R, sep="\n")

**Read the result:** The triangular matrix records how to recombine the two orthonormal directions.

A = QR; R is upper triangular



## 5. Check both guarantees and the projection

Reconstruction and orthonormality are separate requirements. The ambient projection is a third useful consequence.

**Predict first:** Should the 3 × 3 product QQᵀ equal I₃?

- Line 1, `reconstruction_error = np.linalg.norm(A - Q @ R)`: Measures factorization error. Shape: (3, 2) difference → scalar. Operation: ‖A−QR‖ is near machine precision.

- Line 2, `orthogonality_error = np.linalg.norm(Q.T @ Q - np.eye(2))`: Measures loss of orthogonality. Shape: (2, 2) difference → scalar. Operation: ‖QᵀQ−I₂‖ is near machine precision.

- Line 3, `P = Q @ Q.T`: Builds the column-space projection. Shape: (3, 2) @ (2, 3) → (3, 3). Operation: P has rank 2 and differs from I₃.

In [ ]:
reconstruction_error = np.linalg.norm(A - Q @ R)
orthogonality_error = np.linalg.norm(Q.T @ Q - np.eye(2))
P = Q @ Q.T

In [ ]:
print("reconstruction error =", reconstruction_error, "orthogonality error =", orthogonality_error, "P =", P, sep="\n")

**Read the result:** QQᵀ projects onto the reachable plane. Its complement projects onto the leftover normal direction.

P = QQᵀ; P² = P; P ≠ I₃



## 6. Connect the factorization to least squares

The target cannot be fitted exactly, but its Q coordinates can be fitted. Solve Rθ̂ = Qᵀb instead of forming AᵀA.

**Predict first:** Are Qᵀb and θ̂ coefficients in the same basis?

- Line 1, `b = np.ones(3)`: Stores the 4.4 target. Shape: b: (3,). Operation: b = (1,1,1)ᵀ.

- Line 2, `coordinates = Q.T @ b`: Computes coordinates in the q basis. Shape: (2, 3) @ (3,) → (2,). Operation: Qᵀb = (0,4/√6)ᵀ.

- Line 3, `theta_hat = np.linalg.solve(R, coordinates)`: Converts to the original column coefficients. Shape: solve((2, 2), (2,)) → (2,). Operation: θ̂ = (2/3,4/3)ᵀ.

- Line 4, `fitted = A @ theta_hat`: Computes the fitted output. Shape: (3, 2) @ (2,) → (3,). Operation: b̂ = (2/3,4/3,2/3)ᵀ = QQᵀb.

- Line 5, `Q_library, R_library = np.linalg.qr(A, mode="reduced")`: Computes a library factorization. Shape: (3, 2) → (3, 2), (2, 2). Operation: Its signs may differ; verify the product, orthonormality, and projection.

In [ ]:
b = np.ones(3)
coordinates = Q.T @ b
theta_hat = np.linalg.solve(R, coordinates)
fitted = A @ theta_hat
Q_library, R_library = np.linalg.qr(A, mode="reduced")

In [ ]:
print("Q.T @ b =", coordinates, "theta_hat =", theta_hat, "fitted =", fitted, "Q_library =", Q_library, "R_library =", R_library, sep="\n")

**Read the result:** The projected target lives in ℝ³; its q-basis coordinates and model coefficients each live in ℝ² but represent different bases.

Rθ̂ = Qᵀb; Aθ̂ = QQᵀb



## 7. See what changes when a column is dependent

A duplicate direction leaves no remainder. This blocks normalization in the full-column-rank construction, even though QR itself still exists.

**Predict first:** Would dividing this remainder by its norm make a new direction?

- Line 1, `dependent_column = 2. * a1`: Makes a dependent input. Shape: scalar × (3,) → (3,). Operation: The new column is 2a₁ = (−2,0,2)ᵀ.

- Line 2, `w_dependent = dependent_column - (q1 @ dependent_column) * q1`: Subtracts its entire old component. Shape: (3,) → (3,). Operation: The remainder is zero in exact arithmetic.

- Line 3, `threshold = 1e-12 * np.linalg.norm(dependent_column)`: Sets an explicitly scaled numerical tolerance. Shape: scalar. Operation: The threshold is relative to this input’s length.

- Line 4, `no_new_direction = np.linalg.norm(w_dependent) <= threshold`: Detects a vanishing remainder. Shape: scalar comparison → boolean. Operation: no_new_direction = True. Stop before division.

In [ ]:
dependent_column = 2. * a1
w_dependent = dependent_column - (q1 @ dependent_column) * q1
threshold = 1e-12 * np.linalg.norm(dependent_column)
no_new_direction = np.linalg.norm(w_dependent) <= threshold

In [ ]:
print("dependent column =", dependent_column, "remaining vector =", w_dependent, "no new direction =", no_new_direction, sep="\n")

**Read the result:** The column space is only a line. A library QR may complete Q, but the extra column must not be mistaken for another direction in Col(A).

rank([a₁ | 2a₁]) = 1; R is singular

The maintained routine guards the first column and every later remainder; the practical fit uses lstsq if rank is deficient.

## Why Gram–Schmidt produces QR

### Subtracting old components creates a new perpendicular direction

Set wⱼ = aⱼ − Σᵢ<ⱼ(qᵢᵀaⱼ)qᵢ. Pair it with any earlier qₖ. Orthonormality cancels exactly the matching term, leaving qₖᵀwⱼ = 0. If wⱼ were zero, aⱼ would already be in the preceding span, contradicting independence.

Dividing by ‖wⱼ‖₂ gives a unit vector. The same argument applies to the first column with an empty sum: a zero first column must be caught before normalization.

aⱼ = Σᵢ<ⱼ rᵢⱼqᵢ + rⱼⱼqⱼ

### The nested spans make R upper triangular

Each qⱼ is formed using a₁,…,aⱼ, and each aⱼ can be reconstructed using q₁,…,qⱼ. Therefore the two prefixes have the same span. No later direction is needed to reconstruct an earlier column.

Collect these reconstruction coefficients as column j of R. All entries below the diagonal are zero. The nonzero diagonal makes R invertible under the independent-column assumption.

span{a₁,…,aⱼ} = span{q₁,…,qⱼ}; A = QR

### Follow the lecture’s second column exactly

The first normalized column is q₁ = (−1,0,1)ᵀ/√2. The second coefficient is r₁₂ = q₁ᵀa₂ = −1/√2. Subtracting r₁₂q₁ from a₂ = (1,1,0)ᵀ leaves w₂ = (1/2,1,1/2)ᵀ.

Its length is √6/2, so q₂ = (1,2,1)ᵀ/√6. The negative coefficient is allowed: the component of a₂ along q₁ points in the opposite direction.

a₂ = −(1/√2)q₁ + (√6/2)q₂

### Orthonormal coordinates simplify projection and fitting

Because QᵀQ = I₂, the coordinates of the projection of b onto Col(A) are Qᵀb. The fitted vector is QQᵀb. If Aθ̂ equals that fitted vector, then QRθ̂ = QQᵀb, so multiplying by Qᵀ gives Rθ̂ = Qᵀb.

For b = (1,1,1)ᵀ, this produces θ̂ = (2/3,4/3)ᵀ, exactly the 4.4 example. Qᵀb are basis coordinates in q₁,q₂; θ̂ are coefficients of the original columns a₁,a₂. They are not the same vector of numbers.

Qᵀb = (0,4/√6)ᵀ; Rθ̂ = Qᵀb; b̂ = QQᵀb

### Factorization error and orthogonality error test different things

A small ‖A − QR‖ says the factors reconstruct the input. It does not guarantee a small ‖QᵀQ − I‖. Classical Gram–Schmidt can lose orthogonality for near-dependent columns through cancellation, even while reconstructing A accurately.

Modified Gram–Schmidt computes each projection from the updated remainder. It often improves numerical orthogonality, but severe conditioning can still require reorthogonalization. Use a library QR for practical factorization. The companion notebook compares both errors on the original ε = 10⁻⁸ example instead of relying on one allclose check.



### Dependent columns change the full-column-rank algorithm, not the existence of QR

If a₂ = 2a₁, its remainder after subtracting the q₁ component is zero. The teaching routine stops instead of dividing by zero; there is no second independent direction in the column space.

QR still exists: a library may complete Q with an arbitrary perpendicular direction while making R singular. Then using all columns of that Q in QQᵀ can project onto a larger space than Col(A). A rank-r basis Qᵣ gives the correct P = QᵣQᵣᵀ; lstsq handles the fit without requiring an invertible R.



## Numerical checks

These checks verify the example and the correction described above. They support the algebraic arguments and do not replace them.

In [ ]:
assert A.shape == (3, 2) and Q.shape == (3, 2) and R.shape == (2, 2)
assert np.allclose(q1, np.array([-1, 0, 1]) / np.sqrt(2))
assert np.allclose(q2, np.array([1, 2, 1]) / np.sqrt(6))
assert np.allclose(w2, [0.5, 1., 0.5])
assert np.allclose(R, [[np.sqrt(2), -1 / np.sqrt(2)], [0, np.sqrt(6) / 2]])
assert reconstruction_error < 1e-12 and orthogonality_error < 1e-12
assert np.allclose(np.tril(R, -1), 0, rtol=0, atol=1e-12)
assert np.allclose(P.T, P) and np.allclose(P @ P, P)
assert np.allclose(theta_hat, [2 / 3, 4 / 3])
assert np.allclose(fitted, P @ b)
assert np.allclose(Q_library @ R_library, A)
assert np.allclose(Q_library.T @ Q_library, np.eye(2))
assert np.allclose(Q_library @ Q_library.T, P)
assert no_new_direction

def gram_schmidt(matrix, modified=True, rtol=1e-12):
    """Full-column-rank real teaching routine with a scaled remainder guard."""
    matrix = np.asarray(matrix, dtype=float)
    if matrix.ndim != 2 or not np.all(np.isfinite(matrix)):
        raise ValueError("Expected a finite real matrix.")
    m, n = matrix.shape
    if not (m >= n > 0):
        raise ValueError("This teaching routine requires m >= n > 0.")
    threshold = rtol * np.linalg.norm(matrix, 2)
    Q_out = np.zeros((m, n))
    R_out = np.zeros((n, n))
    for j in range(n):
        w = matrix[:, j].copy()
        for i in range(j):
            reference = w if modified else matrix[:, j]
            R_out[i, j] = Q_out[:, i] @ reference
            w -= R_out[i, j] * Q_out[:, i]
        R_out[j, j] = np.linalg.norm(w)
        # This guards j = 0 as well as every later column.
        if R_out[j, j] <= threshold:
            raise ValueError("No new direction at the chosen tolerance.")
        Q_out[:, j] = w / R_out[j, j]
    return Q_out, R_out

Q_mgs, R_mgs = gram_schmidt(A)
assert np.allclose(Q_mgs @ R_mgs, A)
assert np.allclose(Q_mgs.T @ Q_mgs, np.eye(2))
for bad in [np.column_stack((np.zeros(3), a2)), np.column_stack((a1, 2 * a1))]:
    try:
        gram_schmidt(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("A vanishing remainder was not rejected.")
Q_complete, R_complete = np.linalg.qr(A, mode="complete")
assert Q_complete.shape == (3, 3) and R_complete.shape == (3, 2)
assert np.allclose(Q_complete @ Q_complete.T, np.eye(3))
assert np.allclose(Q_complete[:, :2] @ Q_complete[:, :2].T, P)
print("All lecture QR checks passed.")

## Measure reconstruction and orthogonality separately

The original notebook’s small ε example exposes cancellation in classical Gram–Schmidt. Compare classical, modified, and library QR on the same matrix. All reconstruction errors may be small even when the classical Q is far from orthonormal. Modified Gram–Schmidt is still a teaching algorithm; severe conditioning can require reorthogonalization.

In [ ]:
epsilon = 1e-8
A_near = np.array([[1., 1., 1.], [epsilon, 0., 0.], [0., epsilon, 0.], [0., 0., epsilon]])
methods = [("classical", gram_schmidt(A_near, modified=False)),
           ("modified", gram_schmidt(A_near, modified=True)),
           ("library QR", np.linalg.qr(A_near, mode="reduced"))]
errors = []
for name, (Q_near, R_near) in methods:
    reconstruction = np.linalg.norm(A_near - Q_near @ R_near) / np.linalg.norm(A_near)
    orthogonality = np.linalg.norm(Q_near.T @ Q_near - np.eye(3))
    errors.append([reconstruction, orthogonality])
    print(f"{name:12s}: relative reconstruction={reconstruction:.3e}; orthogonality={orthogonality:.3e}")
errors = np.array(errors)
assert np.all(errors[:, 0] < 1e-10)
assert errors[0, 1] > 1e-3
assert errors[1, 1] < errors[0, 1]
assert errors[2, 1] < 1e-12
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.bar([name for name, _ in methods], np.maximum(errors[:, 1], 1e-17))
ax.set(yscale="log", ylabel="Frobenius norm of Q.T @ Q - I", title="Orthogonality errors on the same input")
plt.show()

## Check your understanding

**Must the lecture’s Q match NumPy’s Q entry for entry?**

<details><summary>Show explanation</summary>

No. A column sign in Q can be reversed if the corresponding row of R is reversed. Check A ≈ QR, QᵀQ ≈ I, and the projection instead.

</details>

**Why is R upper triangular?**

<details><summary>Show explanation</summary>

Column aⱼ is already in span{q₁,…,qⱼ}. It has no component along any later qᵢ, so rᵢⱼ = 0 when i > j.

</details>

**What fails when the first column is zero?**

<details><summary>Show explanation</summary>

Its norm is zero, so even the first normalization is invalid. A guard only inside the later-column loop misses this case.

</details>

**Does small A − QR imply QᵀQ is close to I?**

<details><summary>Show explanation</summary>

No. Reconstruction and orthogonality are different properties. Near-dependent columns can expose a large orthogonality error even with a small reconstruction error.

</details>

**Do dependent columns mean there is no QR factorization?**

<details><summary>Show explanation</summary>

No. They mean R is singular and the full-column-rank algorithm cannot make every input column contribute a new direction. Any extra Q columns need not belong to Col(A).

</details>

## Code references

- [NumPy: reduced and complete QR](https://numpy.org/doc/stable/reference/generated/numpy.linalg.qr.html)